# Nagumo travelling front: a heteroclinic orbit

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/19_nagumo_front.ipynb)

u'' + c u' + u(1−u)(u−a) = 0, written as a first-order system. A front joining u = 1 to
u = 0 travels with speed c only for one value of c: there the unstable manifold of the
saddle at (1, 0) lands exactly on the saddle at (0, 0). The exact speed is
c = (1 − 2a)/√2.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.3.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("nagumo_front")
      .ode("u", "v")
      .ode("v", "-c*v-u*(1-u)*(u-a)")
      .parameter(c=0.3, a=0.25)
      .initial(u=0.9, v=0)
      .option(total=40, dt=0.05))
print(mb.render())

u'=v
v'=-c*v-u*(1-u)*(u-a)
par c=0.3,a=0.25
init u=0.9,v=0
@ total=40,dt=0.05
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="plane", x="u", y="v", window=(-0.3, 1.3, -0.5, 0.5),
              sliders={"c": (0.05, 0.9), "a": (0.05, 0.45)}, title="Nagumo front")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Shooting for the speed.** Leave the saddle at (1, 0) along its unstable direction and
see where the orbit goes: below the right c it overshoots u = 0, above it it turns back.
Bisect on c.

In [5]:
import numpy as np
m = ex.m

def miss(c, a=0.25):
    m.pars.update(c=c, a=a)
    e = m.equilibrium({"u": 1, "v": 0})
    lam = max(e.eigenvalues.real)                       # unstable eigenvalue; eigenvector (1, lam)
    tr = m.run(keep=False, u=1 - 1e-6, v=-1e-6 * lam, total=200)
    return 1 if tr["u"].min() < -0.01 else -1           # +1: overshoots

lo, hi = 0.05, 0.9
for _ in range(40):
    mid = 0.5 * (lo + hi)
    lo, hi = (mid, hi) if miss(mid) > 0 else (lo, mid)
print(f"front speed c = {0.5*(lo+hi):.6f}   exact (1-2a)/sqrt(2) = {(1-2*0.25)/np.sqrt(2):.6f}")

front speed c = 0.353553   exact (1-2a)/sqrt(2) = 0.353553


## Things to try

- Change `a`: the speed follows (1 − 2a)/√2 and is 0 at a = 1/2.